# **EDA & Hypothesis Testing**

## Objectives

* Load the cleaned dataset and test four hypotheses about relationships between overtime, income,
  job satisfaction and department, and employee attrition, using descriptive statistics and statistical
  tests to check whether any differences seen in the data are actually significant.
* Answer Business Requirement 1 (attrition overview by department/role) ahead of the hypothesis tests.

## Inputs

* Dataset/CleanData/hr_attrition_clean.csv

## Outputs

* A Summary of Findings table


---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/Users/nav/Desktop/employee-attrition-hackathon/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/Users/nav/Desktop/employee-attrition-hackathon'

### Import Libraries

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg

---

### Import cleaned dataset

I will load the cleaned dataset exported from the ETL notebook

In [6]:
df = pd.read_csv('data/clean/hr_attrition_clean.csv')
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,...,3,4,1,6,3,3,2,2,2,2


## Overview

Before testing any hypotheses. I want to check the balance of the key categorical variables I'll be using, Attrition, OverTime, and Department, since a heavily imbalanced class could affect how I interpret the results later

In [7]:
print("Attrition balance:")
print(df['Attrition'].value_counts(normalize=True).round(3))
print()
print("OverTime balance:")
print(df['OverTime'].value_counts(normalize=True).round(3))
print()
print("Department balance:")
print(df['Department'].value_counts(normalize=True).round(3))

Attrition balance:
Attrition
No     0.839
Yes    0.161
Name: proportion, dtype: float64

OverTime balance:
OverTime
No     0.717
Yes    0.283
Name: proportion, dtype: float64

Department balance:
Department
Research & Development    0.654
Sales                     0.303
Human Resources           0.043
Name: proportion, dtype: float64


Attrition itself is fairly imbalanced, roughly 84% stayed vs 16% left which is worth keeping in mind for the Modelling notebook later where accuracy alone won't tell the full story.

## Business Requirement 1 — Attrition Overview

Before testing the four hypotheses I want to answer the first business requirement directly: what's the overall attrition rate and how does it vary by department and job role?

In [8]:
attrition_rate = (df['Attrition'] == 'Yes').mean()
print(f"Overall attrition rate: {attrition_rate:.1%}")

Overall attrition rate: 16.1%


In [9]:
(pd.crosstab(df['Department'], df['Attrition'], normalize='index') * 100).round(1)

Attrition,No,Yes
Department,,
Human Resources,81.0,19.0
Research & Development,86.2,13.8
Sales,79.4,20.6


In [10]:
(pd.crosstab(df['JobRole'], df['Attrition'], normalize='index') * 100).round(1).sort_values('Yes', ascending=False)

Attrition,No,Yes
JobRole,,
Sales Representative,60.2,39.8
Laboratory Technician,76.1,23.9
Human Resources,76.9,23.1
Sales Executive,82.5,17.5
Research Scientist,83.9,16.1
Healthcare Representative,93.1,6.9
Manufacturing Director,93.1,6.9
Manager,95.1,4.9
Research Director,97.5,2.5


Overall attrition sits at 16.1%. By department it ranges from 13.8% (R&D) to 20.6% (Sales) but the gap is much bigger by job role.Sales Representative sits at 40% while Research Director is only 2.5%.

---

# H1 - Overtime and Attrition

H0: Employee attrition is independent of overtime status.

H1: Employee attrition is associated with overtime status.

In [12]:
df['OverTime'].value_counts()

OverTime
No     1054
Yes     416
Name: count, dtype: int64

I am going to use pg.chi2_independence() to do the Chi-Square Test. The arguments we use are data x and y as the variables for the chi-squared test. y is the target variable Attrition analysed across the feature OverTime.

In [13]:
expected, observed, stats = pg.chi2_independence(data=df, x='OverTime', y='Attrition')

The test summary (stats) has the result of the Pearson Chi-Square test.

In [14]:
stats

,test,lambda,chi2,dof,pval,cramer,power
0,pearson,1.000000,87.564294,1.0,8.158424e-21,0.244065,1.0
1,cressie-read,0.666667,84.576205,1.0,3.696877e-20,0.239864,1.0
2,log-likelihood,0.000000,80.079543,1.0,3.596366e-19,0.233401,1.0
3,freeman-tukey,-0.500000,77.830595,1.0,1.122677e-18,0.230100,1.0
4,mod-log-likelihood,-1.000000,76.421211,1.0,2.291765e-18,0.228007,1.0
5,neyman,-2.000000,75.828553,1.0,3.093947e-18,0.227121,1.0


To get the p-value from the Pearson test. I will query from stats where test == 'pearson' and grab the p-value.

In [15]:
stats.query("test == 'pearson'")['pval']

0    8.158424e-21
Name: pval, dtype: float64

Consider our significance level alpha = 0.05.

The p-value (8.16e-21) is far smaller than alpha. This means that we reject the null hypothesis.

This means there is a significant association between OverTime and Attrition.

Result: Rejected H0 (p < 0.001)

Summary: Employees who work overtime attrite at 30.5%, nearly three times the rate of employees who don't
(10.4%). This is by far the largest effect of any hypothesis tested in this notebook.

# H2 - Monthly Income and Attrition

H0: There is no significant difference in Monthly Income between employees who leave and those who stay.

H1: There is a significant difference in Monthly Income between employees who leave and those who stay.

First, a look at the mean, median and standard deviation of MonthlyIncome for leavers vs stayers. This shows whether employees who leave tend to earn less on average, and whether that pattern is consistent or spread out across the group.

In [16]:
df.groupby('Attrition')['MonthlyIncome'].agg(['mean', 'median', 'std'])

,mean,median,std
Attrition,,,
No,6832.739659,5204.0,4818.208001
Yes,4787.092827,3202.0,3640.210367


This suggests employees who leave earn noticeably less on average (mean £4,787 vs £6,833), with the median showing an even bigger gap (£3,202 vs £5,204) meaning the difference isn't just being driven by a few high earners staying on.

Before running a t-test I need to check whether MonthlyIncome is normally distributed for both groups since a t-test assumes roughly normal data. 

In [17]:
leaver = df[df['Attrition']=='Yes']['MonthlyIncome']
stayer = df[df['Attrition']=='No']['MonthlyIncome']

pg.normality(data=df, dv='MonthlyIncome', group='Attrition', alpha=0.05)

,W,pval,normal
Attrition,,,
Yes,0.779897,1.502541e-17,False
No,0.834134,5.970379e-34,False


Both groups fail the normality test (p < 0.05 for both) so MonthlyIncome is not normally distributed for either group.

I am doing the t-test to check whether the difference in MonthlyIncome between the two groups is statistically significant.

In [18]:
pg.ttest(leaver, stayer)

,T,dof,alternative,p-val,CI95%,cohen-d,BF10,power
T-test,-7.482622,412.740748,two-sided,4.433589e-13,"[-2583.05, -1508.24]",0.440018,4.471e+10,0.999989


Now I am going to pull the p-value from the test result to check it against the 0.05 significance threshold.

In [19]:
pg.ttest(leaver, stayer).loc['T-test', 'p-val']

4.433588628286071e-13

Significance level alpha = 0.05.

The p-value is far smaller than alpha. This means that we reject the null hypothesis.

This means there is a significant difference in Monthly Income between employees who leave and those who stay.

Result: Rejected H0 (p < 0.001)

Summary: Employees who leave earn noticeably less on average (mean £4,787, median £3,202) than employees
who stay (mean £6,833, median £5,204). This difference is statistically significant confirming income is
a genuine driver of attrition not just something that looks different by chance.

# H3 - Job Satisfaction and Attrition

H0: There is no significant difference in Job Satisfaction between employees who leave and those who stay.

H1: There is a significant difference in Job Satisfaction between employees who leave and those who stay.

First a look at the mean, median and standard deviation of JobSatisfaction for leavers vs stayers.

In [21]:
df.groupby('Attrition')['JobSatisfaction'].agg(['mean', 'median', 'std'])

,mean,median,std
Attrition,,,
No,2.778589,3.0,1.093277
Yes,2.468354,3.0,1.118058


This suggests employees who leave report slightly lower job satisfaction on average (mean 2.47 vs 2.78), though the medians are the same for both groups (3.0), so this looks like a smaller less clear-cut gap than H1 or H2.

Before running a t-test I need to check whether JobSatisfaction is normally distributed for both groups.

In [22]:
leaver = df[df['Attrition']=='Yes']['JobSatisfaction']
stayer = df[df['Attrition']=='No']['JobSatisfaction']

pg.normality(data=df, dv='JobSatisfaction', group='Attrition', alpha=0.05)

,W,pval,normal
Attrition,,,
Yes,0.850929,2.311557e-14,False
No,0.842537,2.875281e-33,False


I am doing the t-test to check whether the difference in JobSatisfaction between the two groups is statistically significant, in the same way as H2 despite the normality test failing here too.

In [23]:
pg.ttest(leaver, stayer)

,T,dof,alternative,p-val,CI95%,cohen-d,BF10,power
T-test,-3.926113,328.593446,two-sided,0.000105,"[-0.47, -0.15]",0.282725,148.213,0.978497


In [24]:
pg.ttest(leaver, stayer).loc['T-test', 'p-val']

0.0001052049107397441

Significance level alpha = 0.05.

The p-value is smaller than alpha. This means that we reject the null hypothesis.

This means there is a significant difference in Job Satisfaction between employees who leave and those who stay.

Result: Rejected H0 (p < 0.001)

Summary: Employees who leave report lower job satisfaction on average (mean 2.47) than employees who stay
(mean 2.78). 

# H4 — Department and Attrition

H0: Employee attrition is independent of department.

H1: Employee attrition is associated with department.

In [25]:
df['Department'].value_counts()

Department
Research & Development    961
Sales                     446
Human Resources            63
Name: count, dtype: int64

 I am going to use pg.chi2_independence() to do the Chi-Square Test. y is the target variable Attrition analysed across the feature Department.

In [26]:
expected, observed, stats = pg.chi2_independence(data=df, x='Department', y='Attrition')

In [27]:
stats

,test,lambda,chi2,dof,pval,cramer,power
0,pearson,1.000000,10.796007,2.0,0.004526,0.085698,0.845592
1,cressie-read,0.666667,10.686054,2.0,0.004781,0.085261,0.841692
2,log-likelihood,0.000000,10.490345,2.0,0.005273,0.084477,0.834544
3,freeman-tukey,-0.500000,10.363646,2.0,0.005618,0.083965,0.829771
4,mod-log-likelihood,-1.000000,10.253226,2.0,0.005937,0.083516,0.825517
5,neyman,-2.000000,10.078297,2.0,0.006479,0.082801,0.818594


In [28]:
stats.query("test == 'pearson'")['pval']

0    0.004526
Name: pval, dtype: float64

Consider our significance level alpha = 0.05.

The p-value (0.0045) is smaller than alpha. This means that we reject the null hypothesis.

This means there is a significant association between Department and Attrition.

Result: Rejected H0 (p = 0.005)

Summary: Sales (20.6%) and HR (19.0%) attrite more than R&D (13.8%). This is a real statistically
significant effect but the smallest of the four hypotheses tested.

# Summary of Findings

H1 — Overtime | Rejected H0 | Employees on overtime attrite at 30.5% vs 10.4% for those not on overtime the largest effect found

H2 — Monthly Income | Rejected H0 | Leavers earn less on average (£4,787 vs £6,833)

H3 — Job Satisfaction | Rejected H0 | Leavers report lower satisfaction on average (2.47 vs 2.78) a smaller effect than H1/H2

H4 — Department | Rejected H0 | Sales and HR attrite more than R&D the smallest but still significant effect

All four hypotheses are supported by the data. Note that MonthlyIncome and JobSatisfaction both failed
normality testing but the t-test was used regardless

---